In [2]:
import numpy as np

# States and observations
states = ['h', 'e', 'l', 'o']            # S1..S4
observations = ['O1', 'O2', 'O3', 'O4']  # O1..O4
obs_sequence = [0, 1, 2, 3]              # sequence: O1, O2, O3, O4

# Transition matrix A (from rows -> to cols)
A = np.array([
    [0.0, 0.7, 0.3, 0.0],   # S1 (/h/)
    [0.0, 0.2, 0.6, 0.2],   # S2 (/e/)
    [0.0, 0.0, 0.3, 0.7],   # S3 (/l/)
    [0.0, 0.0, 0.1, 0.9],   # S4 (/o/)
])

# Emission matrix B: rows states, cols observations
B = np.array([
    [0.6, 0.2, 0.1, 0.1],   # S1 (/h/)
    [0.1, 0.7, 0.1, 0.1],   # S2 (/e/)
    [0.1, 0.1, 0.6, 0.2],   # S3 (/l/)
    [0.2, 0.1, 0.2, 0.5],   # S4 (/o/)
])

# Initial probabilities (start at /h/)
pi = np.array([1.0, 0.0, 0.0, 0.0])

def viterbi(A, B, pi, obs):
    N = A.shape[0]
    T = len(obs)

    delta = np.zeros((T, N))
    psi = np.zeros((T, N), dtype=int)

    # Initialization t = 0
    for i in range(N):
        delta[0, i] = pi[i] * B[i, obs[0]]
        psi[0, i] = 0

    # Recursion
    for t in range(1, T):
        for j in range(N):
            prev_probs = delta[t-1, :] * A[:, j]      # probabilities from each prev state -> j
            psi[t, j] = int(np.argmax(prev_probs))    # best previous state index
            delta[t, j] = prev_probs[psi[t, j]] * B[j, obs[t]]

    # Termination
    last_state = int(np.argmax(delta[T-1, :]))
    best_prob = delta[T-1, last_state]

    # Backtrack
    path_idx = [last_state]
    for t in range(T-1, 0, -1):
        path_idx.insert(0, psi[t, path_idx[0]])

    path_states = [states[i] for i in path_idx]
    return path_states, best_prob, delta, psi

# Run
best_path, best_prob, delta, psi = viterbi(A, B, pi, obs_sequence)

# Print nicely
np.set_printoptions(precision=6, suppress=True)
print("Observation sequence:", [observations[i] for i in obs_sequence], "\n")

print("Delta matrix (rows = time steps t=0..3, cols = states [h,e,l,o]):")
print(delta, "\n")

print("Psi (backpointers) matrix (same indexing):")
print(psi, "\n")

print("Most likely phoneme sequence:", best_path)
print("Probability of that sequence (joint prob): {:.6f}".format(best_prob), "\n")

# Short inference
print("Inference:")
if best_path == ['h','e','l','o']:
    print("Viterbi correctly decodes the phoneme sequence as /h/ -> /e/ -> /l/ -> /o/ for observations [O1,O2,O3,O4].")
else:
    print("Decoded sequence differs; check matrices or observation inputs.")


Observation sequence: ['O1', 'O2', 'O3', 'O4'] 

Delta matrix (rows = time steps t=0..3, cols = states [h,e,l,o]):
[[0.6      0.       0.       0.      ]
 [0.       0.294    0.018    0.      ]
 [0.       0.00588  0.10584  0.01176 ]
 [0.       0.000118 0.00635  0.037044]] 

Psi (backpointers) matrix (same indexing):
[[0 0 0 0]
 [0 0 0 0]
 [0 1 1 1]
 [0 1 2 2]] 

Most likely phoneme sequence: ['h', 'e', 'l', 'o']
Probability of that sequence (joint prob): 0.037044 

Inference:
Viterbi correctly decodes the phoneme sequence as /h/ -> /e/ -> /l/ -> /o/ for observations [O1,O2,O3,O4].


Because the HMM transition and emission probabilities strongly favour the natural progression
ℎ
→
𝑒
→
𝑙
→
𝑜
h→e→l→o and each observation matches its respective phoneme with high emission probability, the Viterbi algorithm decodes the observation sequence

[O1,O2,O3,O4] as the phoneme sequence /h/ → /e/ → /l/ → /o/ with joint probability 0.037044. This confirms the HMM recogniser correctly identifies the word "hello" from the given acoustic features.